# Checkout support: Lightning judges where each request goes

**Lightning = routing judge and efficient executor. GPT Sol 6.1 = capable executor.**

The judge forecasts whether Lightning can complete the next stage. Switchyard's built-in `llm_task_classifier` selects the executor from that forecast. Every judge call is included in the cost comparison.

Run all cells with the defaults to replay the saved real responses. **No API key or inference calls are needed in recorded mode.**

In [ ]:
from pathlib import Path
import sys, json, hashlib, statistics
import pandas as pd
from IPython.display import display, Image, Markdown, HTML

_start = Path.cwd().resolve()
ROOT = next(p for p in [_start, *_start.parents] if (p / 'shared/incident_demo.py').is_file())
EXAMPLE = ROOT / 'examples/checkout-support/lightning-gpt-sol-judge'
CHECKOUT = EXAMPLE.parent
sys.path.insert(0, str(ROOT / 'shared'))
sys.path.insert(0, str(CHECKOUT))

In [ ]:
DEMO_MODE = "recorded"  # recorded | live
ALLOW_PAID_CALLS = False
SELECTED_REPEAT = 1  # 1, 2 or 3
REPEATS = 1  # only applies to a fresh live run
RECORDING = ROOT / 'recordings/checkout-support/lightning-gpt-sol-judge-v3-2026-10-17-20261017T081627323689Z.json'
assert DEMO_MODE in {'recorded', 'live'}
assert SELECTED_REPEAT in {1, 2, 3}

## The customer's incident

The screen and payment backend disagree. A duplicate webhook must not be counted as a second financial capture. No payment, refund or repair is performed by this demo.

In [ ]:
case = json.loads((CHECKOUT / 'case.json').read_text())
display(Markdown('> ' + case['ticket']))
display(Image(filename=str(CHECKOUT / 'checkout.png'), width=840))
display(pd.DataFrame([
    {'Backend evidence': 'Settled capture', 'Observed value': 'cap-42 · SGD 118.70 · one unique capture'},
    {'Backend evidence': 'Order state', 'Observed value': case['payment']['order_state']},
    {'Backend evidence': 'Repeated webhook', 'Observed value': 'Same capture ID; not another charge'},
    {'Backend evidence': 'Confirmation request', 'Observed value': 'HTTP 504'},
    {'Backend evidence': 'Runbook', 'Observed value': case['runbook']['id']},
]))

## Load the recording—or explicitly opt into fresh calls

Recorded mode reads all nine saved attempts, including the failed Lightning-only workflows. Live mode uses the same workflow and TOML profile through your own configured gateway. Changing both controls above to `DEMO_MODE = "live"` and `ALLOW_PAID_CALLS = True` enables paid calls; otherwise no inference is sent.

Fresh results are written under ignored `runs/checkout-support/`. The pinned workshop recording is preserved. Tracing is not enabled by this workflow.

In [ ]:
if DEMO_MODE == 'live':
    if not ALLOW_PAID_CALLS:
        raise ValueError('Live inference requires explicit ALLOW_PAID_CALLS = False.')
    from dotenv import load_dotenv
    import importlib.util
    sys.path.insert(0, str(EXAMPLE))
    spec = importlib.util.spec_from_file_location('lightning_judge_run', EXAMPLE / 'run.py')
    judge_run = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(judge_run)
    load_dotenv(ROOT / '.env', override=True)
    active_recording = await judge_run.compare()
else:
    active_recording = RECORDING

recording = json.loads(active_recording.read_text())
results = recording['results']
rates = recording['metadata']['rates']['rates']
if DEMO_MODE == 'recorded':
    source_map = json.loads((ROOT / 'evidence/source-map.json').read_text())
    for original_path, digest in recording['metadata']['source_hashes'].items():
        measured = ROOT / source_map[original_path][digest]
        assert hashlib.sha256(measured.read_bytes()).hexdigest() == digest
assert all(r['cost_complete'] for r in results), 'Missing usage prevents a complete cost comparison.'
print(f'{DEMO_MODE.upper()} · {len(results)} task attempts · no hidden retries or excluded failures')

Costs below use ordinary input/output prices for every token, **excluding cache discounts and write premiums**. This keeps the workshop comparison focused on model routing. These are reference-price estimates, not actual bills. The observed cache-adjusted estimates are available in the expandable technical details.

Lightning-only costs cover **partial failed workflows**, not a completed solution. Every arm has the same two-attempt limit per stage.

In [ ]:
from IPython.display import HTML
LABELS = {'super_only': 'GPT Sol 6.1 only', 'lightning_only': 'Lightning only', 'routed': 'Lightning judge + routed executors'}

def ordinary_input_cost(result):
    return sum((c['usage']['input_tokens'] * rates[c['model']]['input_per_million']
                + c['usage']['output_tokens'] * rates[c['model']]['output_per_million']) / 1_000_000
               for c in result['calls'])

rows = []
for arm, label in LABELS.items():
    attempts = [r for r in results if r['arm'] == arm]
    accepted = sum(r['passed'] for r in attempts)
    total = sum(r['cost_usd'] for r in attempts)
    rows.append({'Approach': label, 'Accepted': f'{accepted}/{len(attempts)}',
                 'Mean estimated USD/workflow': total / len(attempts),
                 'USD/accepted including failures': total / accepted if accepted else None,
                 'Median seconds': statistics.median(r['elapsed_seconds'] for r in attempts),
                 'Model calls': sum(len(r['calls']) for r in attempts),
                 'Judge calls': sum(c['kind'] == 'routing_judge' for r in attempts for c in r['calls']),
                 'Ordinary-input repricing USD/workflow': statistics.mean(ordinary_input_cost(r) for r in attempts)})
summary = pd.DataFrame(rows)
# One pricing basis in the main table and headline.
base, routed = rows[0], rows[2]
saving = 1 - routed['Mean estimated USD/workflow'] / base['Mean estimated USD/workflow']
ordinary_saving = 1 - routed['Ordinary-input repricing USD/workflow'] / base['Ordinary-input repricing USD/workflow']
presentation = summary[['Approach', 'Accepted', 'Ordinary-input repricing USD/workflow', 'Median seconds', 'Model calls', 'Judge calls']].rename(
    columns={'Ordinary-input repricing USD/workflow': 'Estimated USD/workflow (cache discounts excluded)'})
display(presentation.style.format({'Estimated USD/workflow (cache discounts excluded)': '${:.6f}', 'Median seconds': '{:.2f}'}))
if all(r['passed'] for r in results if r['arm'] in {'super_only', 'routed'}):
    display(Markdown(f'**{ordinary_saving:.1%} lower estimated model cost**, including judge calls and excluding cache effects.'))
else:
    display(Markdown('**Some GPT-only or routed tasks failed. Partial-workflow spending is not a completed-solution savings result.**'))
display(HTML('<details><summary>Technical cost details: observed cache behavior</summary>'
    + f'<p>With the cache reads and writes reported in this run, the reference-price reduction is {saving:.1%}. '
      'This changes with cache state and is not the workshop headline. These are estimates, not gateway bills.</p>'
    + summary.to_html(index=False, float_format=lambda x: f'{x:.6f}') + '</details>'))

## See Lightning judge, then see the chosen executor

For each stage, inspect Lightning's `p_solve` forecast, explanation and selected executor. Switchyard uses a 0.75 threshold; this is a routing parameter, not an image-to-GPT rule. Forecasts are not calibrated success guarantees.

Change `SELECTED_REPEAT` above to inspect all three recorded repeats. The call ledger includes both judge and executor requests.

In [ ]:
selected = next(r for r in results if r['arm'] == 'routed' and r['repeat'] == SELECTED_REPEAT)
decisions = []
for decision in selected['judge_decisions']:
    verdict = decision.get('verdict') or {}
    decisions.append({'Stage': decision['stage'], 'Lightning p_solve': verdict.get('p_solve'),
        'Selected executor': 'Lightning' if decision['selected_tier'] == 'weak' else 'GPT Sol 6.1',
        'Judge explanation': verdict.get('crux', 'Invalid judge response: see raw recording')})
display(Markdown('### Lightning made these routing decisions'))
display(pd.DataFrame(decisions))
steps = []
for c in selected['calls']:
    steps.append({'Step': len(steps) + 1, 'Task': c['kind'], 'Role': c.get('role', 'executor'),
        'Model': c['model'].rsplit('/', 1)[-1], 'Routing reason': c['decision_reason'],
        'Image sent': c['has_image'], 'Estimated USD (cache discounts excluded)': ordinary_input_cost({'calls': [c]}),
        'Seconds': c['seconds']})
display(pd.DataFrame(steps).style.format({'Estimated USD (cache discounts excluded)': '${:.6f}', 'Seconds': '{:.2f}'}))
display(Markdown(f"**Repeat {SELECTED_REPEAT}:** {'accepted' if selected['passed'] else 'not accepted'} · "
                 f"${ordinary_input_cost(selected):.6f} estimated, cache discounts excluded · {selected['elapsed_seconds']:.2f} seconds"))
display(Markdown('### Proposed customer reply\n\n' + (selected['customer_reply'] or 'No accepted reply.')))
display(pd.Series(selected['report'] or {}, name='Validated support report').to_frame())

## How the judge makes the choice

The judge sees the actual task text, model capability descriptions, attachment metadata and any validator rejection feedback. It does not see image pixels or fixture answers. The selected executor receives the original payload, including the actual screenshot.

Lightning forecasts success; Switchyard applies the threshold and selects the target. Python validators evaluate the structured task output and can request one retry, which Lightning judges again. There is no hard-coded image-to-GPT or rejected-output-to-GPT selection. A wrong text-only choice for an image is blocked and counted as a failure.

The task still has three fixed stages; this changes routing, not workflow planning. All final routed attempts used Lightning for text and GPT for the image, without retries.

In [ ]:
partial = []
for r in results:
    if r['arm'] == 'lightning_only':
        partial.append({'Repeat': r['repeat'], 'Accepted': r['passed'],
            'Stop reason': r['error_type'],
            'Detail': '; '.join(b['reason'] for b in r['blocked_requests']),
            'Partial estimated USD (cache effects excluded)': ordinary_input_cost(r)})
display(pd.DataFrame(partial).sort_values('Repeat'))
price_rows = [{'Model': model.rsplit('/', 1)[-1], **price} for model, price in rates.items()]
display(pd.DataFrame(price_rows).set_index('Model').style.format('${:.2f}'))

## Workshop explanation

“Lightning is both our routing judge and our efficient worker. Before each stage it predicts whether it can handle the request. Switchyard selects the executor, and the cost table includes the judge's own tokens.”

Use the cost reduction and completion counts shown above. The final pinned comparison passed 3/3 with both GPT-only and routing, with **68.8% lower estimated model cost including judge calls, excluding cache effects**. These are reference prices applied to observed token counts, not invoices or a new cold-cache run.

Earlier attempts exposed overconfidence after record-ID extraction failures. The final output contract explicitly distinguishes log record IDs from payment object IDs, identically for all baselines. Earlier results remain in the findings; they are not pooled with the final run. One synthetic case repeated three times is not a general quality benchmark.

[Full findings and earlier trials](README.md) · [Judge implementation](judge.py) · [Model profile](models.toml) · [Workshop guide](../../../README.md)